In [1]:
import h5py
import numpy as np
import sys
import os

In [2]:
expert_id_dir = "expert_ids"
if not os.path.exists(expert_id_dir):
    os.makedirs(expert_id_dir)

In [3]:
def expert_operator_id_from_mh_dataset(dataset_path):
    """ 
    For robomimic mh dataset.
    """
    operator_keys = ['better_operator_1', 'better_operator_2', 'okay_operator_1', 'okay_operator_2' , 'worse_operator_1', 'worse_operator_2']
    with h5py.File(dataset_path, 'r') as hdf5_file:

        demo_name2_operator_id = {}
        for operator_id, operator in enumerate(operator_keys):
            operator_demos = [b.decode('utf-8') for b in hdf5_file['mask'][operator]]
            for demo_name in operator_demos:
                demo_name2_operator_id[demo_name] = operator_id
    return demo_name2_operator_id

def save_expert_ids(output_file,dataset_path, demo_name2_operator_id):
    demo_names = list(demo_name2_operator_id.keys()) 
    demo_names.sort(key=lambda x: int(x.split('_')[-1]))

    
    with open(output_file, 'w') as f:
        f.write(f"dataset_path: {dataset_path}\n")
        for demo_name in demo_names:
            operator_id = demo_name2_operator_id[demo_name]
            f.write(f"{demo_name}:{operator_id}\n") 

    print(f"Expert IDs saved to {output_file}")
    print(f"Total expert demos: {len(demo_names)}")
    print(f"Operator IDs: {set(demo_name2_operator_id.values())}")

# load the expert ids as a dictionary
def load_expert_ids(file_path):
    expert_ids = {}
    dataset_path = None
    with open(file_path, 'r') as f:
        #first line is the dataset path
        first_line = f.readline().strip()
        if first_line.startswith("dataset_path:"):
            dataset_path = first_line.split(":")[1].strip()
        # subsequent lines are demo_name:operator_id
        for line in f: 
            demo_name, operator_id = line.strip().split(':')
            expert_ids[demo_name] = int(operator_id)
    return expert_ids

In [4]:
dataset_path = "/home/carl/offline_study/robomimic/datasets/can/mh/low_dim.hdf5"
demo_name2_operator_id = expert_operator_id_from_mh_dataset(dataset_path)
save_expert_ids(expert_id_dir+"/expert_ids_can_mh_lowdim.txt", dataset_path, demo_name2_operator_id)

Expert IDs saved to expert_ids/expert_ids_can_mh_lowdim.txt
Total expert demos: 300
Operator IDs: {0, 1, 2, 3, 4, 5}


In [6]:
expert_ids = load_expert_ids(expert_id_dir+"/expert_ids_can_mh_lowdim.txt")
print(f"Loaded {len(expert_ids)} expert IDs from file.")
print(f"Dataset path: {dataset_path}")
print(f"Example expert ID: {list(expert_ids.items())[0]}")

Loaded 300 expert IDs from file.
Dataset path: /home/carl/offline_study/robomimic/datasets/can/mh/low_dim.hdf5
Example expert ID: ('demo_0', 0)
